# Tree Starts Notebook

This notebook will be used to train and evaluate Random Forest models. The Random Forest model will first be run without being corrected, and then run after preprocessing to compare the results.
This was done after initial work and so is missing some explatory steps of this work (all other files will be saved in an appendix folder).

In the notebook you will find the below:

1. [Data and EDA](#read-in-data-and-eda)
2. [Setting up the model](#setting-up-the-model)
3. [Baseline model](#baseline-model)
4. [Hyperparameter tuning](#hyperparameter-tuning)
5. [Comparing the results](#results-comparison)


The different data sources:
1. `../data/target/model_training_data_final_with_lc.csv` 
2. `../data/target/model_training_data_final_urban.csv`
3. `../data/target/model_training_data_final_urban_only.csv`
4. `../data/target/model_training_data_final.csv`

Read in the necessary libraries:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor, HistGradientBoostingRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, root_mean_squared_error, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, roc_curve,precision_score, recall_score
import seaborn as sns
import warnings
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler
import networkx as nx
from itertools import product
import pickle
import lightgbm as lgb
import xgboost as xgb
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.exceptions import DataConversionWarning
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVR
import matplotlib.colors as mcolors
import matplotlib.cm as cm
plt.rcParams['font.family'] = 'Helvetica'
warnings.filterwarnings('ignore', category=DataConversionWarning)
%matplotlib inline

%load_ext autoreload
%autoreload 2

from helper_functions import *

Read in the already cleaned and split data:

In [3]:
# read in complex filled data
df = pd.read_csv('../data/target/model_training_data_final.csv')
X_train_c, X_test_c, y_train_c, y_test_c = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex')

# read in simple filled data
X_train_s, X_test_s, y_train_s, y_test_s = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='simple')


Quick baseline models with both sets:

In [ ]:
# trying three random other models to see how they compare

# lightgbm
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_c, y_train_c)

scores = cross_val_score(rf_b_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_rfb = rf_b_model.predict(X_test_c)
test_r2_rfb = r2_score(y_test_c, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_c, y_train_c)

scores = cross_val_score(ex_b_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_exb = ex_b_model.predict(X_test_c)
test_r2_exb = r2_score(y_test_c, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train_c, y_train_c)

scores = cross_val_score(lgb_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_lgb = lgb_model.predict(X_test_c)
test_r2_lgb = r2_score(y_test_c, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")

print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train_c, y_train_c)

scores = cross_val_score(hgbr_model, X_train_c, y_train_c, cv=5, scoring='r2',verbose=0)
print(f"mean CV score: {scores.mean():.4f}")

y_pred_hgbr = hgbr_model.predict(X_test_c)
test_r2_hgbr = r2_score(y_test_c, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")

print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
br_model.fit(X_train_c, y_train_c)

scores = cross_val_score(br_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_br = br_model.predict(X_test_c)
test_r2_br = r2_score(y_test_c, y_pred_br)
print(f"Test Set R²: {test_r2_br:.4f}")

print("for GradientBoostingRegressor:")
gbr_model = GradientBoostingRegressor(random_state=42,verbose=0)
gbr_model.fit(X_train_c, y_train_c)

scores = cross_val_score(gbr_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_gbr = gbr_model.predict(X_test_c)
test_r2_gbr = r2_score(y_test_c, y_pred_gbr)
print(f"Test Set R²: {test_r2_gbr:.4f}")

print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
xgb_b_model.fit(X_train_c, y_train_c)

scores = cross_val_score(xgb_b_model, X_train_c, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_xgbb = xgb_b_model.predict(X_test_c)
test_r2_xgbb = r2_score(y_test_c, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

# SVR requires scaling
print("for SVR:")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

svr_model = SVR(kernel='rbf')
svr_model.fit(X_train_scaled, y_train_c.values.ravel())

scores = cross_val_score(svr_model, X_train_scaled, y_train_c.values.ravel(), cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_svr = svr_model.predict(X_test_scaled)
print(f"Test R²: {r2_score(y_test_c, y_pred_svr):.4f}")

#Results:
# for RandomForest:
# mean CV score: 0.2572
# Test Set R²: 0.3340
# for ExtraTrees:
# mean CV score: 0.1586
# Test Set R²: 0.3038
# for LGBM:
# mean CV score: 0.2382
# Test Set R²: 0.3056
# for HistGradientBoostingRegressor:
# mean CV score: 0.2364
# Test Set R²: 0.2871
# for BaggingRegressor:
# mean CV score: 0.1889
# Test Set R²: 0.2291
# for GradientBoostingRegressor:
# mean CV score: 0.1506
# Test Set R²: 0.1648
# for XGBRegressor:
# mean CV score: 0.1675
# Test Set R²: 0.2617
# for SVR:
# mean CV score: -0.0099
# Test R²: -0.0106

for RandomForest:
mean CV score: 0.2572
Test Set R²: 0.3340
for ExtraTrees:
mean CV score: 0.1586
Test Set R²: 0.3038
for LGBM:
mean CV score: 0.2382
Test Set R²: 0.3056
for HistGradientBoostingRegressor:
mean CV score: 0.2364
Test Set R²: 0.2871
for BaggingRegressor:
mean CV score: 0.1889
Test Set R²: 0.2291
for GradientBoostingRegressor:
mean CV score: 0.1506
Test Set R²: 0.1648
for XGBRegressor:
mean CV score: 0.1675
Test Set R²: 0.2617
for SVR:
mean CV score: -0.0099
Test R²: -0.0106


now for the simple filling:

In [ ]:
# trying three random other models to see how they compare

# lightgbm
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_s, y_train_s)

scores = cross_val_score(rf_b_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_rfb = rf_b_model.predict(X_test_s)
test_r2_rfb = r2_score(y_test_s, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_s, y_train_s)

scores = cross_val_score(ex_b_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_exb = ex_b_model.predict(X_test_s)
test_r2_exb = r2_score(y_test_s, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train_s, y_train_s)

scores = cross_val_score(lgb_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_lgb = lgb_model.predict(X_test_s)
test_r2_lgb = r2_score(y_test_s, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")

print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train_s, y_train_s)

scores = cross_val_score(hgbr_model, X_train_s, y_train_s, cv=5, scoring='r2',verbose=0)
print(f"mean CV score: {scores.mean():.4f}")

y_pred_hgbr = hgbr_model.predict(X_test_s)
test_r2_hgbr = r2_score(y_test_s, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")

print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
br_model.fit(X_train_s, y_train_s)

scores = cross_val_score(br_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_br = br_model.predict(X_test_s)
test_r2_br = r2_score(y_test_s, y_pred_br)
print(f"Test Set R²: {test_r2_br:.4f}")

print("for GradientBoostingRegressor:")
gbr_model = GradientBoostingRegressor(random_state=42,verbose=0)
gbr_model.fit(X_train_s, y_train_s)

scores = cross_val_score(gbr_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_gbr = gbr_model.predict(X_test_s)
test_r2_gbr = r2_score(y_test_s, y_pred_gbr)
print(f"Test Set R²: {test_r2_gbr:.4f}")

print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
xgb_b_model.fit(X_train_s, y_train_s)

scores = cross_val_score(xgb_b_model, X_train_s, y_train_s, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_xgbb = xgb_b_model.predict(X_test_s)
test_r2_xgbb = r2_score(y_test_s, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

# SVR requires scaling
print("for SVR:")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_s)
X_test_scaled = scaler.transform(X_test_s)

svr_model = SVR(kernel='rbf')
svr_model.fit(X_train_scaled, y_train_s.values.ravel())

scores = cross_val_score(svr_model, X_train_scaled, y_train_c.values.ravel(), cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_svr = svr_model.predict(X_test_scaled)
print(f"Test R²: {r2_score(y_test_s, y_pred_svr):.4f}")

# Results:
# for RandomForest:
# mean CV score: -0.0073
# Test Set R²: 0.0899
# for ExtraTrees:
# mean CV score: -0.0934
# Test Set R²: 0.1405
# for LGBM:
# mean CV score: 0.1386
# Test Set R²: 0.2331
# for HistGradientBoostingRegressor:
# mean CV score: 0.1247
# Test Set R²: 0.1989
# for BaggingRegressor:
# mean CV score: -0.0811
# Test Set R²: 0.0347
# for GradientBoostingRegressor:
# mean CV score: 0.1026
# Test Set R²: 0.1100
# for XGBRegressor:
# mean CV score: 0.0709
# Test Set R²: 0.1540
# for SVR:
# mean CV score: -0.0105
# Test R²: -0.0113

for RandomForest:
mean CV score: -0.0073
Test Set R²: 0.0899
for ExtraTrees:
mean CV score: -0.0934
Test Set R²: 0.1405
for LGBM:
mean CV score: 0.1386
Test Set R²: 0.2331
for HistGradientBoostingRegressor:
mean CV score: 0.1247
Test Set R²: 0.1989
for BaggingRegressor:
mean CV score: -0.0811
Test Set R²: 0.0347
for GradientBoostingRegressor:
mean CV score: 0.1026
Test Set R²: 0.1100
for XGBRegressor:
mean CV score: 0.0709
Test Set R²: 0.1540
for SVR:
mean CV score: -0.0105
Test R²: -0.0113


Ok, best base model here was: _RandomForest_ (with LGBM right behind). 

The complex filled preformed better, and we saw this in the earlier work, so we'll go forward with that filling. Next, we'll do some cleaning of the data etc to see how we can get the grided model to go, and then will do the ungridded model after. 

First, lets run a dummy model to have this compare to: 

In [ ]:
dummy = DummyRegressor(strategy='median')

dummy_r2 = cross_val_score(dummy, X_train_c, y_train_c, scoring='r2', cv=5)
print(f"Baseline R²: {dummy_r2.mean():.4f}")

# Also check on test set
dummy.fit(X_train_c, y_train_c)
y_pred_dummy = dummy.predict(X_test_c)
print(f"Baseline Test R²: {r2_score(y_test_c, y_pred_dummy):.4f}")

print_consistent_results(dummy, X_train_c, y_train_c, X_test_c, y_test_c, cv=5)

# Baseline R²: -0.0155
# Baseline Test R²: -0.0162
# Cross-validation R²:   -0.0155 ± 0.0012
# Cross-validation RMSE: 4.9939 ± 0.4396
# Test Set R²:           -0.0162
# Test Set RMSE:         5.2641

Baseline R²: -0.0155
Baseline Test R²: -0.0162
Cross-validation R²:   -0.0155 ± 0.0012
Cross-validation RMSE: 4.9939 ± 0.4396
Test Set R²:           -0.0162
Test Set RMSE:         5.2641


Ok, inital model, this one we'll definitly need to get rid of some of the data becuase it reduced it's accuracy with some of the new additions:

In [ ]:
model_init_1 = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1,
)

print("results for: model_training_data_final.csv")
print_consistent_results(model_init_1, X_train_c, y_train_c, X_test_c, y_test_c, cv=5)

# results for: model_training_data_final.csv
# Cross-validation R²:   0.2593 ± 0.0177
# Cross-validation RMSE: 4.2611 ± 0.3398
# Test Set R²:           0.3314
# Test Set RMSE:         4.2699

results for: model_training_data_new_2.csv
Cross-validation R²:   0.2593 ± 0.0177
Cross-validation RMSE: 4.2611 ± 0.3398
Test Set R²:           0.3314
Test Set RMSE:         4.2699


Showing result comparions with scaling of features before selecting for best features:

In [27]:
scaler = StandardScaler()
pipeline_init_1 = Pipeline([('s', scaler), ('m', model_init_1)])

print_consistent_results(pipeline_init_1, X_train_c, y_train_c, X_test_c, y_test_c, cv=5)

Cross-validation R²:   0.2599 ± 0.0182
Cross-validation RMSE: 4.2595 ± 0.3417
Test Set R²:           0.3302
Test Set RMSE:         4.2739


In [28]:
binary_cols = [c for c in X_train_c.columns if X_train_c[c].nunique() == 2]
continuous_cols = [c for c in X_train_c.columns if c not in binary_cols]

preprocessor = ColumnTransformer([
    ('scale', PowerTransformer(), continuous_cols),
    ('passthrough', 'passthrough', binary_cols)
])

pipeline_init_2 = Pipeline([('pre', preprocessor), ('m', model_init_1)])

print_consistent_results(pipeline_init_2, X_train_c, y_train_c, X_test_c, y_test_c, cv=5)

Cross-validation R²:   0.2452 ± 0.0352
Cross-validation RMSE: 4.2965 ± 0.3015
Test Set R²:           0.3221
Test Set RMSE:         4.2996


In [30]:
scaler = MinMaxScaler()
pipeline_init_2 = Pipeline([('s',scaler),('m',model_init_1)])

# cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
# # evaluate the model
# # m_scores = cross_val_score(pipeline, X_train_w_grid, y_train, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# # rmse_scores = (-m_scores) ** 0.5
# # print(f"RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}")

# r2_scores_scale = cross_val_score(pipeline, X_train_w_grid, y_train, scoring='r2', cv=cv, n_jobs=-1)
# print(f"R2:   {r2_scores_scale.mean():.4f} ± {r2_scores_scale.std():.4f}")

print_consistent_results(pipeline_init_2, X_train_c, y_train_c, X_test_c, y_test_c, cv=5)

Cross-validation R²:   0.2552 ± 0.0243
Cross-validation RMSE: 4.2710 ± 0.3245
Test Set R²:           0.3280
Test Set RMSE:         4.2807


Select best columns for model:

In [40]:
groups = find_correlated_groups(X_train_c,thresh=0.80)

cols_init_full = select_best_features(X_train_c, y_train_c, groups, RandomForestRegressor(random_state=42,n_jobs=-1), 
                         method='spearman', large_group_thresh=5, cv=5,must_include='grid_point_id')

Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
Large group (9 features) → selected: mean_available
Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
Subsampled to 200 combinations
Searching 200 combinations from 10 small groups...
Best exhaustive combo score: 0.2964

Final feature count: 22 from original 68


In [ ]:
pd.Series(cols_init_full).to_csv('cols_init_full.csv', index=False)

**Run a model now with all of the new selected columns**

In [ ]:
model_init_2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1,
)

print_consistent_results(model_init_2, X_train_c[cols_init_full], y_train_c, X_test_c[cols_init_full], y_test_c, cv=5)

# Cross-validation R²:   0.2947 ± 0.0130
# Cross-validation RMSE: 4.1599 ± 0.3500
# Test Set R²:           0.3756
# Test Set RMSE:         4.1264

Cross-validation R²:   0.2947 ± 0.0130
Cross-validation RMSE: 4.1599 ± 0.3500
Test Set R²:           0.3756
Test Set RMSE:         4.1264


Quick grid search for this model:

Note to self, this seems like the best preforming one. We will add the logistic regression results to this one to see if it improves the results at all.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}

grid_search_w_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1, # use all CPU cores
    verbose=1  # print progress
)

grid_search_w_grid.fit(X_train_c[cols_init_full], y_train_c)

print(f"Best Parameters: {grid_search_w_grid.best_params_}")
print(f"Best CV R²:      {grid_search_w_grid.best_score_:.4f}")

# Use the best model on the test set
best_model_w_grid = grid_search_w_grid.best_estimator_
print_consistent_results(best_model_w_grid, X_train_c[cols_init_full], y_train_c, X_test_c[cols_init_full], y_test_c, cv=5)

# Fitting 5 folds for each of 324 candidates, totalling 1620 fits
# Best Parameters: {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
# Best CV R²:      0.2998
# Cross-validation R²:   0.2998 ± 0.0186
# Cross-validation RMSE: 4.1447 ± 0.3544
# Test Set R²:           0.3714
# Test Set RMSE:         4.1402

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best Parameters: {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best CV R²:      0.2998
Cross-validation R²:   0.2998 ± 0.0186
Cross-validation RMSE: 4.1447 ± 0.3544
Test Set R²:           0.3714
Test Set RMSE:         4.1402


I'll do the super simple method of dropping the columns that are least correlated with our target to see how the results compare to the more time intenive method.

In [ ]:
corr_cols_to_drop = drop_correlated_features(X_train_c, y_train_c, 'spearman', 0.9)
X_train_c_simp_select = X_train_c.drop(columns=corr_cols_to_drop)
X_test_c_simp_select = X_test_c.drop(columns=corr_cols_to_drop)

model_init_2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1,
)

print_consistent_results(model_init_2, X_train_c_simp_select, y_train_c, X_test_c_simp_select, y_test_c, cv=5)

# with thresh of 0.85
# Cross-validation R²:   0.2602 ± 0.0182
# Cross-validation RMSE: 4.2575 ± 0.3260
# Test Set R²:           0.3236
# Test Set RMSE:         4.2949

# #thresh of 0.8
# Cross-validation R²:   0.2569 ± 0.0186
# Cross-validation RMSE: 4.2668 ± 0.3267
# Test Set R²:           0.3227
# Test Set RMSE:         4.2976

# thresh of 0.9
# Cross-validation R²:   0.2622 ± 0.0198
# Cross-validation RMSE: 4.2520 ± 0.3285
# Test Set R²:           0.3141
# Test Set RMSE:         4.3247

Cross-validation R²:   0.2622 ± 0.0198
Cross-validation RMSE: 4.2520 ± 0.3285
Test Set R²:           0.3141
Test Set RMSE:         4.3247


**Filter out this data now based on data availability or affectedness and see how the model preforms**

first the availability threshold: 

In [ ]:
cols_init_full_with_avail = cols_init_full.copy()
cols_init_full_with_avail.append('perc_available_upscaled_new')

In [ ]:
for thr in [0.1,0.2,0.3,0.4]:
    try_out_models_for_col_thresh(X_train_c[cols_init_full_with_avail], X_test_c[cols_init_full_with_avail],
                                  y_train_c, y_test_c, best_model_w_grid,col='perc_available_upscaled_new',
                                  thresh=thr)
    
# mean CV score: 0.2724401765606063 with thresh: 0.1
# R²: 0.3215  RMSE: 5.2312
# mean CV score: 0.1404177084817387 with thresh: 0.2
# R²: 0.4941  RMSE: 3.6945
# mean CV score: -1.7047414976482949 with thresh: 0.3
# R²: -6.5844  RMSE: 0.1982
# mean CV score: -20519.421245887464 with thresh: 0.4
# R²: -0.0136  RMSE: 0.0001

mean CV score: 0.2724401765606063 with thresh: 0.1
R²: 0.3215  RMSE: 5.2312
mean CV score: 0.1404177084817387 with thresh: 0.2
R²: 0.4941  RMSE: 3.6945
mean CV score: -1.7047414976482949 with thresh: 0.3
R²: -6.5844  RMSE: 0.1982
mean CV score: -20519.421245887464 with thresh: 0.4
R²: -0.0136  RMSE: 0.0001


In [ ]:
for thr in [0.1,0.2,0.3,0.4]:
    try_out_models_for_col_thresh(X_train_c[cols_init_full_with_avail], X_test_c[cols_init_full_with_avail],
                                  y_train_c, y_test_c, RandomForestRegressor(random_state=42,n_jobs=-1),
                                  'perc_available_upscaled_new',thresh=thr)

# mean CV score: 0.2650168912265605 with thresh: 0.1
# R²: 0.3308  RMSE: 5.1950
# mean CV score: 0.10602393626226073 with thresh: 0.2
# R²: 0.4712  RMSE: 3.7770
# mean CV score: -1.901925028515818 with thresh: 0.3
# R²: -8.5620  RMSE: 0.2226
# mean CV score: -21482.21289607342 with thresh: 0.4
# R²: -0.0147  RMSE: 0.0001

mean CV score: 0.2650168912265605 with thresh: 0.1
R²: 0.3308  RMSE: 5.1950
mean CV score: 0.10602393626226073 with thresh: 0.2
R²: 0.4712  RMSE: 3.7770
mean CV score: -1.901925028515818 with thresh: 0.3
R²: -8.5620  RMSE: 0.2226
mean CV score: -21482.21289607342 with thresh: 0.4
R²: -0.0147  RMSE: 0.0001


In [ ]:
models = [ExtraTreesRegressor(random_state=42,n_jobs=-1),lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0),
          HistGradientBoostingRegressor(random_state=42,verbose=0),BaggingRegressor(random_state=42, n_jobs=-1, verbose=0),
          GradientBoostingRegressor(random_state=42,verbose=0),xgb.XGBRegressor(random_state=42)]
for i,model in enumerate(models):
    print(f"new model {i}")
    for thr in [0.1,0.2,0.3,0.4]:
        try_out_models_for_col_thresh(X_train_c[cols_init_full_with_avail], X_test_c[cols_init_full_with_avail],
                                    y_train_c, y_test_c,model,
                                    col='perc_available_upscaled_new',thresh=thr)

# best preforming when we look at the balance between the two R2:        
# new model 2
# mean CV score: 0.2488854880154978 with thresh: 0.1
# R²: 0.2641  RMSE: 5.4476

Now the affected threshold:

In [ ]:
for thr in [0.1,0.2,0.3,0.4]:
    try_out_models_for_col_thresh(X_train_c[cols_init_full_with_avail], X_test_c[cols_init_full_with_avail],
                                  y_train_c, y_test_c, best_model_w_grid,col='pot_affected_perc',
                                  thresh=thr)

# mean CV score: 0.09520106908691092 with thresh: 0.1
# R²: 0.1504  RMSE: 5.4415
# mean CV score: 0.12660458796432866 with thresh: 0.2
# R²: 0.1324  RMSE: 5.1572
# mean CV score: -0.2811977722903928 with thresh: 0.3
# R²: -0.0960  RMSE: 5.8595
# mean CV score: -2.086060088290534 with thresh: 0.4
# R²: -7.4436  RMSE: 2.0887

mean CV score: 0.09520106908691092 with thresh: 0.1
R²: 0.1504  RMSE: 5.4415
mean CV score: 0.12660458796432866 with thresh: 0.2
R²: 0.1324  RMSE: 5.1572
mean CV score: -0.2811977722903928 with thresh: 0.3
R²: -0.0960  RMSE: 5.8595
mean CV score: -2.086060088290534 with thresh: 0.4
R²: -7.4436  RMSE: 2.0887


In [ ]:
for thr in [0.1,0.2,0.01,0.05]:
    try_out_models_for_col_thresh(X_train_c[cols_init_full_with_avail], X_test_c[cols_init_full_with_avail],
                                  y_train_c, y_test_c,RandomForestRegressor(random_state=42,n_jobs=-1),
                                  col='pot_affected_perc',thresh=thr)

# mean CV score: 0.12258652434868293 with thresh: 0.1
# R²: 0.1722  RMSE: 5.3715
# mean CV score: 0.13469344727123114 with thresh: 0.2
# R²: 0.1302  RMSE: 5.1637
# mean CV score: 0.16668614550025854 with thresh: 0.01
# R²: 0.3064  RMSE: 5.6107
# mean CV score: 0.09631310731329586 with thresh: 0.05
# R²: 0.2577  RMSE: 5.5709

mean CV score: 0.12258652434868293 with thresh: 0.1
R²: 0.1722  RMSE: 5.3715
mean CV score: 0.13469344727123114 with thresh: 0.2
R²: 0.1302  RMSE: 5.1637
mean CV score: 0.16668614550025854 with thresh: 0.01
R²: 0.3064  RMSE: 5.6107
mean CV score: 0.09631310731329586 with thresh: 0.05
R²: 0.2577  RMSE: 5.5709


In [ ]:
models = [ExtraTreesRegressor(random_state=42,n_jobs=-1),lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0),
          HistGradientBoostingRegressor(random_state=42,verbose=0),BaggingRegressor(random_state=42, n_jobs=-1, verbose=0),
          GradientBoostingRegressor(random_state=42,verbose=0),xgb.XGBRegressor(random_state=42)]
for i,model in enumerate(models):
    print(f"new model {i}")
    for thr in [0.1,0.2,0.01,0.05]:
        try_out_models_for_col_thresh(X_train_c[cols_init_full], X_test_c[cols_init_full],
                                    y_train_c, y_test_c,model,
                                    col='pot_affected_perc',thresh=thr)

# new model 2
# mean CV score: 0.17119704545974784 with thresh: 0.1
# R²: 0.1502  RMSE: 5.4422

# new model 2
# mean CV score: 0.17323311174030517 with thresh: 0.1
# R²: 0.1502  RMSE: 5.4422

Ok, when we look at the grided data, it seems like keeping the data whole is the best option for this model.

When balancing both the CV and test R2 and RSME's we see the gridsearch and filtered data preforming the best. Next we're just going to add in the logistic regression predictions in to see if these help the results at all.

**Look into the baselines without the `grid_point_id`:**

Now, we need to move on to what feels like the more important question here, on how useful can our NTL be on it's own without the spatial pattern and information the grid_point_id is giving. Now, we need to drop the columns from the data and see which model preforms the best here.

In [4]:
#without grid
X_train_wo_grid = X_train_c.drop(columns=['grid_point_id'])
X_test_wo_grid = X_test_c.drop(columns=['grid_point_id'])

we'll start with the overall search again just to make sure this is the correct model to go forward with:

In [ ]:
# trying three random other models to see how they compare

# lightgbm
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(rf_b_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_rfb = rf_b_model.predict(X_test_wo_grid)
test_r2_rfb = r2_score(y_test_c, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(ex_b_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_exb = ex_b_model.predict(X_test_wo_grid)
test_r2_exb = r2_score(y_test_c, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(lgb_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_lgb = lgb_model.predict(X_test_wo_grid)
test_r2_lgb = r2_score(y_test_c, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")

print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(hgbr_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2',verbose=0)
print(f"mean CV score: {scores.mean():.4f}")

y_pred_hgbr = hgbr_model.predict(X_test_wo_grid)
test_r2_hgbr = r2_score(y_test_c, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")

print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
br_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(br_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_br = br_model.predict(X_test_wo_grid)
test_r2_br = r2_score(y_test_c, y_pred_br)
print(f"Test Set R²: {test_r2_br:.4f}")

print("for GradientBoostingRegressor:")
gbr_model = GradientBoostingRegressor(random_state=42,verbose=0)
gbr_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(gbr_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_gbr = gbr_model.predict(X_test_wo_grid)
test_r2_gbr = r2_score(y_test_c, y_pred_gbr)
print(f"Test Set R²: {test_r2_gbr:.4f}")

print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
xgb_b_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(xgb_b_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_xgbb = xgb_b_model.predict(X_test_wo_grid)
test_r2_xgbb = r2_score(y_test_c, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

print("for XGBRegressor with hist:")
xgb_b_model = xgb.XGBRegressor(random_state=42,tree_method='hist')
xgb_b_model.fit(X_train_wo_grid, y_train_c)

scores = cross_val_score(xgb_b_model, X_train_wo_grid, y_train_c, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_xgbb = xgb_b_model.predict(X_test_wo_grid)
test_r2_xgbb = r2_score(y_test_c, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

#Results:
# for RandomForest:
# mean CV score: -0.0098
# Test Set R²: -0.0484
# for ExtraTrees:
# mean CV score: -0.0612
# Test Set R²: -0.1003
# for LGBM:
# mean CV score: 0.0885
# Test Set R²: 0.1014
# for HistGradientBoostingRegrebssor:
# mean CV score: 0.0975
# Test Set R²: 0.1046
# for BaggingRegressor:
# mean CV score: -0.0602
# Test Set R²: -0.1155
# for GradientBoostingRegressor:
# mean CV score: 0.0820
# Test Set R²: 0.0679
# for XGBRegressor:
# mean CV score: -0.0106
# Test Set R²: 0.0030
# for XGBRegressor with hist:
# mean CV score: -0.0106
# Test Set R²: 0.0030
# for SVR:
# mean CV score: -0.0103
# Test R²: -0.0110

for XGBRegressor:
mean CV score: -0.0106
Test Set R²: 0.0030
for XGBRegressor with hist:
mean CV score: -0.0106
Test Set R²: 0.0030


In [8]:
df_b = pd.read_csv('../data/target/model_training_data_final_with_lc.csv')
X_train_b, X_test_b, y_train_b, y_test_b = clean_and_split_data(df_b,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex')

print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_b, y_train_b)

scores = cross_val_score(rf_b_model, X_train_b, y_train_b, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_lgb = rf_b_model.predict(X_test_b)
test_r2_lgb = r2_score(y_test_b, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")

print_consistent_results(rf_b_model, X_train_b, y_train_b, X_test_b, y_test_b, cv=5)


for RandomForest:
mean CV score: 0.2226
Test Set R²: 0.2568
Cross-validation R²:   0.2226 ± 0.0337
Cross-validation RMSE: 4.3615 ± 0.3169
Test Set R²:           0.2568
Test Set RMSE:         4.5019


In [6]:
df_b = pd.read_csv('../data/target/model_training_data_final_with_lc.csv')
X_train_b, X_test_b, y_train_b, y_test_b = clean_and_split_data(df_b,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='complex')

X_train_wo_grid_b = X_train_b.drop(columns=['grid_point_id'])
X_test_wo_grid_b = X_test_b.drop(columns=['grid_point_id'])

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train_wo_grid_b, y_train_b)

scores = cross_val_score(lgb_model, X_train_wo_grid_b, y_train_b, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_lgb = lgb_model.predict(X_test_wo_grid_b)
test_r2_lgb = r2_score(y_test_b, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")

print_consistent_results(lgb_model, X_train_wo_grid_b, y_train_b, X_test_wo_grid_b, y_test_b, cv=5)


for LGBM:
mean CV score: 0.0944
Test Set R²: 0.1174
Cross-validation R²:   0.0944 ± 0.0219
Cross-validation RMSE: 4.7105 ± 0.3629
Test Set R²:           0.1174
Test Set RMSE:         4.9060


Where we don't have the `grid_point_id`, it looks like maybe: `HistGradientBoostingRegressor` and `LGBM` are preforming best.

In [ ]:
param_grid_hgb = {
    "max_iter":        [100, 200, 300],
    "max_depth":       [3, 5, 7, None],
    "learning_rate":   [0.01, 0.05, 0.1],
    "min_samples_leaf":[10, 20, 50],
    "l2_regularization":[0.0, 0.1, 1.0],
}

grid_hgb_wo = GridSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_grid_hgb,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_hgb_wo.fit(X_train_wo_grid, y_train_c)
best_hgb_wo_grid = grid_hgb_wo.best_estimator_

print("Best R²:    ", grid_hgb_wo.best_score_.round(3))
print("Best params:", grid_hgb_wo.best_params_)

print_consistent_results(best_hgb_wo_grid, X_train_wo_grid, y_train_c, X_test_wo_grid, y_test_c, cv=5)

# results:
# Best R²:     0.113
# Best params: {'l2_regularization': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'max_iter': 100, 'min_samples_leaf': 10}
# Cross-validation R²:   0.1126 ± 0.0326
# Cross-validation RMSE: 4.6640 ± 0.3864
# Test Set R²:           0.1128
# Test Set RMSE:         4.9188

In [ ]:
param_grid_lgbm = {
    "n_estimators":   [100, 200, 300],
    "max_depth":      [3, 5, 7, -1],
    "learning_rate":  [0.01, 0.05, 0.1],
    "num_leaves":     [31, 63, 127],
    "min_child_samples": [10, 20, 50],
    "reg_lambda":     [0.0, 0.1, 1.0],
}

grid_lgbm_wo = GridSearchCV(
    lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_grid_lgbm,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_lgbm_wo.fit(X_train_wo_grid, y_train_c)
best_lgbm_wo_grid = grid_lgbm_wo.best_estimator_

print("Best R²:    ", grid_lgbm_wo.best_score_.round(3))
print("Best params:", grid_lgbm_wo.best_params_)

print_consistent_results(best_lgbm_wo_grid, X_train_wo_grid, y_train_c, X_test_wo_grid, y_test_c, cv=5)

# results:
# Best R²:     0.109
# Best params: {'learning_rate': 0.01, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 31, 'reg_lambda': 1.0}
# Cross-validation R²:   0.1091 ± 0.0341
# Cross-validation RMSE: 4.6755 ± 0.4129
# Test Set R²:           0.1165
# Test Set RMSE:         4.9086

**Time to do some feature selection for our gridless data**

In [ ]:
groups_wo = find_correlated_groups(X_train_wo_grid,thresh=0.85)

cols_wo_grid_hgb = select_best_features(X_train_wo_grid, y_train_c, groups_wo, HistGradientBoostingRegressor(random_state=42,n_jobs=-1), 
                                         method='spearman', large_group_thresh=5, cv=5)

pd.Series(cols_wo_grid_hgb).to_csv('cols_wo_grid_hgb.csv', index=False)

# Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
# Large group (9 features) → selected: mean_available
# Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
# Subsampled to 200 combinations
# Searching 200 combinations from 10 small groups...
# Best exhaustive combo score: 0.0989

# Final feature count: 21 from original 67

# time: 34m 2.2s

In [ ]:
#groups are the same here as above
cols_wo_grid_lgbm = select_best_features(X_train_wo_grid, y_train_c, groups_wo, lgb.LGBMRegressor(random_state=42, n_jobs=-1), 
                                         method='spearman', large_group_thresh=5, cv=5)

pd.Series(cols_wo_grid_lgbm).to_csv('cols_wo_grid_lgbm.csv', index=False)

# Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
# Large group (9 features) → selected: mean_available
# Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
# Subsampled to 200 combinations
# Searching 200 combinations from 10 small groups...
# Best exhaustive combo score: 0.0890

# Final feature count: 21 from original 67

# time: 11m 35.6s

change the threshold:

In [ ]:
groups_wo_2 = find_correlated_groups(X_train_wo_grid,thresh=0.85)

cols_wo_grid_hgb_2 = select_best_features(X_train_wo_grid, y_train_c, groups_wo_2, HistGradientBoostingRegressor(random_state=42), 
                                         method='spearman', large_group_thresh=5, cv=5)

pd.Series(cols_wo_grid_hgb_2).to_csv('cols_wo_grid_hgb_2.csv', index=False)

# Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
# Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
# Subsampled to 200 combinations
# Searching 200 combinations from 12 small groups...
# Best exhaustive combo score: 0.1004

# Final feature count: 22 from original 67

# time: 5m 8.6s

In [ ]:
groups_wo_3 = find_correlated_groups(X_train_wo_grid,thresh=0.9)

cols_wo_grid_hgb_3 = select_best_features(X_train_wo_grid, y_train_c, groups_wo_3, HistGradientBoostingRegressor(random_state=42), 
                                         method='spearman', large_group_thresh=5, cv=5)

pd.Series(cols_wo_grid_hgb_3).to_csv('cols_wo_grid_hgb_3.csv', index=False)

# Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
# Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
# Subsampled to 200 combinations
# Searching 200 combinations from 12 small groups...
# Best exhaustive combo score: 0.0976

# Final feature count: 25 from original 67
# time: 4. 41.2s

In [ ]:
#groups are the same here as above
cols_wo_grid_lgbm_2 = select_best_features(X_train_wo_grid, y_train_c, groups_wo_2, lgb.LGBMRegressor(random_state=42, n_jobs=-1), 
                                         method='spearman', large_group_thresh=5, cv=5)

pd.Series(cols_wo_grid_lgbm_2).to_csv('cols_wo_grid_lgbm_2.csv', index=False)

# Large group (18 features) → selected: prop_not_enough_data_impact_dur_pers_avail30
# Large group (8 features) → selected: mean_recov_dur_pers_avail30_isna
# Subsampled to 200 combinations
# Searching 200 combinations from 12 small groups...
# Best exhaustive combo score: 0.0924

# Final feature count: 22 from original 67

# time: 11m 44s

Ok, now that we have these sets of columns, lets see how they do with our models:

First set:

In [ ]:
# best_hgb_wo_grid.fit(X_train_wo_grid[cols_wo_grid_hgb], y_train_c)
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid[cols_wo_grid_hgb], y_train_c,
                          X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# results:
# Cross-validation R²:   0.1105 ± 0.0329
# Cross-validation RMSE: 4.6670 ± 0.3547
# Test Set R²:           0.0932
# Test Set RMSE:         4.9729

In [ ]:
# best_lgbm_wo_grid.fit(X_train_wo_grid[cols_wo_grid_lgbm], y_train_c)
print_consistent_results(best_lgbm_wo_grid, X_train_wo_grid[cols_wo_grid_lgbm], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_lgbm], y_test_c, cv=5)

# results:
# Cross-validation R²:   0.1006 ± 0.0359
# Cross-validation RMSE: 4.6966 ± 0.4024
# Test Set R²:           0.0925
# Test Set RMSE:         4.9746

Second set:

In [ ]:
# best_hgb_wo_grid.fit(X_train_wo_grid[cols_wo_grid_hgb], y_train_c)
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid[cols_wo_grid_hgb], y_train_c,
                          X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# results:
# Cross-validation R²:   0.0955 ± 0.0391
# Cross-validation RMSE: 4.7054 ± 0.3570
# Test Set R²:           0.0856
# Test Set RMSE:         4.9936


In [ ]:
# best_hgb_wo_grid.fit(X_train_wo_grid[cols_wo_grid_hgb], y_train_c)
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid[cols_wo_grid_hgb_3], y_train_c,
                          X_test_wo_grid[cols_wo_grid_hgb_3], y_test_c, cv=5)

# results
# Cross-validation R²:   0.0975 ± 0.0347
# Cross-validation RMSE: 4.7018 ± 0.3681
# Test Set R²:           0.1055
# Test Set RMSE:         4.9389

In [ ]:
# best_lgbm_wo_grid.fit(X_train_wo_grid[cols_wo_grid_lgbm_2], y_train_c)
print_consistent_results(best_lgbm_wo_grid, X_train_wo_grid[cols_wo_grid_lgbm_2], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_lgbm_2], y_test_c, cv=5)

# results:
# Cross-validation R²:   0.1030 ± 0.0235
# Cross-validation RMSE: 4.6900 ± 0.3846
# Test Set R²:           0.1058
# Test Set RMSE:         4.9382

Very simple feature selection:

In [ ]:
corr_cols_to_drop_wo = drop_correlated_features(X_train_wo_grid, y_train_c, 'spearman', 0.85)
X_train_c_simp_selec_wo = X_train.drop(columns=corr_cols_to_drop)
X_test_c_simp_select_wo = X_test.drop(columns=corr_cols_to_drop)

In [ ]:
# best_hgb_wo_grid.fit(X_train_c_simp_selec_wo,y_train_c)
print_consistent_results(best_hgb_wo_grid, X_train_c_simp_selec_wo, y_train_c, 
                         X_test_c_simp_select_wo, y_test_c, cv=5)

# results:
# Cross-validation R²:   0.1045 ± 0.0233
# Cross-validation RMSE: 4.6858 ± 0.3829
# Test Set R²:           0.0894
# Test Set RMSE:         4.9830

In [ ]:
# best_lgbm_wo_grid.fit(X_train_c_simp_selec_wo,y_train_c)
print_consistent_results(best_lgbm_wo_grid, X_train_c_simp_selec_wo, y_train_c, 
                         X_test_c_simp_select_wo, y_test_c, cv=5)

# results:
# Cross-validation R²:   0.1074 ± 0.0377
# Cross-validation RMSE: 4.6801 ± 0.4169
# Test Set R²:           0.1051
# Test Set RMSE:         4.9401

In [ ]:
for thr in [0.7,0.75,0.8,0.85,0.9]:
    print(thr)
    corr_cols_to_drop_wo = drop_correlated_features(X_train_wo_grid, y_train_c, 'spearman', thr)
    X_train_c_simp_selec_wo = X_train_wo_grid.drop(columns=corr_cols_to_drop_wo)
    X_test_c_simp_select_wo = X_test_wo_grid.drop(columns=corr_cols_to_drop_wo)

    best_hgb_wo_grid.fit(X_train_c_simp_selec_wo, y_train_c)
    y_pred = best_hgb_wo_grid.predict(X_test_c_simp_select_wo)
    
    # Test set metrics
    test_r2 = r2_score(y_test_c, y_pred)
    test_rmse = root_mean_squared_error(y_test_c, y_pred)

    print(f"Test Set R²:           {test_r2:.4f}")
    print(f"Test Set RMSE:         {test_rmse:.4f}")

#results:
# 0.7
# Test Set R²:           0.0900
# Test Set RMSE:         4.9816
# 0.75
# Test Set R²:           0.0851
# Test Set RMSE:         4.9949
# 0.8
# Test Set R²:           0.0894
# Test Set RMSE:         4.9830
# 0.85
# Test Set R²:           0.0894
# Test Set RMSE:         4.9830
# 0.9
# Test Set R²:           0.0891
# Test Set RMSE:         4.9839

In [ ]:
for thr in [0.7,0.75,0.8,0.85,0.9]:
    print(thr)
    corr_cols_to_drop_wo = drop_correlated_features(X_train_wo_grid, y_train_c, 'spearman', thr)
    X_train_c_simp_selec_wo = X_train_wo_grid.drop(columns=corr_cols_to_drop_wo)
    X_test_c_simp_select_wo = X_test_wo_grid.drop(columns=corr_cols_to_drop_wo)

    best_lgbm_wo_grid.fit(X_train_c_simp_selec_wo, y_train_c)
    y_pred = best_lgbm_wo_grid.predict(X_test_c_simp_select_wo)
    
    # Test set metrics
    test_r2 = r2_score(y_test_c, y_pred)
    test_rmse = root_mean_squared_error(y_test_c, y_pred)

    print(f"Test Set R²:           {test_r2:.4f}")
    print(f"Test Set RMSE:         {test_rmse:.4f}")
    print(f"\n")

# Results:
# 0.7
# Test Set R²:           0.0965
# Test Set RMSE:         4.9636
#0.75
# Test Set R²:           0.1021
# Test Set RMSE:         4.9484
#0.8
# Test Set R²:           0.1051
# Test Set RMSE:         4.9401
#0.85
# Test Set R²:           0.1051
# Test Set RMSE:         4.9401
#0.9
# Test Set R²:           0.1089
# Test Set RMSE:         4.9296

In [ ]:
corr_cols_to_drop_wo = drop_correlated_features(X_train_wo_grid, y_train_c, 'spearman', 0.9)
X_train_c_simp_selec_wo = X_train_wo_grid.drop(columns=corr_cols_to_drop_wo)
X_test_c_simp_select_wo = X_test_wo_grid.drop(columns=corr_cols_to_drop_wo)

# best_lgbm_wo_grid.fit(X_train_c_simp_selec_wo,y_train_c)
print_consistent_results(best_lgbm_wo_grid, X_train_c_simp_selec_wo, y_train_c, 
                         X_test_c_simp_select_wo, y_test_c, cv=5)

# Cross-validation R²:   0.1109 ± 0.0412
# Cross-validation RMSE: 4.6701 ± 0.4136
# Test Set R²:           0.1089
# Test Set RMSE:         4.9296

Depending on these results we'll try to do a gridsearch for these:

In [ ]:
param_grid_hgb = {
    "max_iter":        [100, 200, 300],
    "max_depth":       [3, 5, 7, None],
    "learning_rate":   [0.01, 0.05, 0.1],
    "min_samples_leaf":[10, 20, 50],
    "l2_regularization":[0.0, 0.1, 1.0],
}

grid_hgb_wo = GridSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_grid_hgb,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_hgb_wo.fit(X_train_wo_grid[cols_wo_grid_hgb], y_train_c)
best_hgb_wo_grid = grid_hgb_wo.best_estimator_

print("Best R²:    ", grid_hgb_wo.best_score_.round(3))
print("Best params:", grid_hgb_wo.best_params_)

print_consistent_results(best_hgb_wo_grid, X_train_wo_grid[cols_wo_grid_hgb], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# results:
# Best R²:     0.118
# Best params: {'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_depth': 5, 'max_iter': 100, 'min_samples_leaf': 10}
# Cross-validation R²:   0.1184 ± 0.0259
# Cross-validation RMSE: 4.6483 ± 0.3693
# Test Set R²:           0.0951
# Test Set RMSE:         4.9675

In [ ]:
# Compare feature sets — what got dropped?
dropped_corrs_1 = {}
dropped = set(X_train_wo_grid.columns) - set(cols_wo_grid_hgb_3)
for col in sorted(dropped):
    corr = X_train_wo_grid[col].corr(y_train_c, method='spearman')
    dropped_corrs_1[col] = corr

dropped_corrs_1_sort = dict(sorted(dropped_corrs_1.items(), key=lambda x: x[1],reverse=True))
dropped_corrs_1_sort

In [ ]:
dropped_corrs = {}
dropped = set(X_train_wo_grid.columns) - set(X_train_c_simp_selec_wo.columns)
for col in sorted(dropped):
    corr = X_train_wo_grid[col].corr(y_train_c, method='spearman')
    dropped_corrs[col] = corr

dropped_corrs_sort = dict(sorted(dropped_corrs.items(), key=lambda x: x[1],reverse=True))

Just wanted to check if there were any columns that really shouldn't have been excluded:

In [ ]:
param_grid_lgbm = {
    "n_estimators":   [100, 200, 300],
    "max_depth":      [3, 5, 7, -1],
    "learning_rate":  [0.01, 0.05, 0.1],
    "num_leaves":     [31, 63, 127],
    "min_child_samples": [10, 20, 50],
    "reg_lambda":     [0.0, 0.1, 1.0],
}

grid_lgbm_wo = GridSearchCV(
    lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_grid_lgbm,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_lgbm_wo.fit(X_train_c_simp_selec_wo, y_train_c)
best_lgbm_wo_grid = grid_lgbm_wo.best_estimator_

print("Best R²:    ", grid_lgbm_wo.best_score_.round(3))
print("Best params:", grid_lgbm_wo.best_params_)

print_consistent_results(best_lgbm_wo_grid, X_train_c_simp_selec_wo, y_train_c, 
                         X_test_c_simp_select_wo, y_test_c, cv=5)

# results:
# Best R²:     0.111
# Best params: {'learning_rate': 0.01, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 31, 'reg_lambda': 1.0}
# Cross-validation R²:   0.1109 ± 0.0412
# Cross-validation RMSE: 4.6701 ± 0.4136
# Test Set R²:           0.1089
# Test Set RMSE:         4.9296

Now, let's filter out data by availability and affected data

In [ ]:
models = [ExtraTreesRegressor(random_state=42,n_jobs=-1),lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0),
          HistGradientBoostingRegressor(random_state=42,verbose=0),BaggingRegressor(random_state=42, n_jobs=-1, verbose=0),
          GradientBoostingRegressor(random_state=42,verbose=0),xgb.XGBRegressor(random_state=42)]
for i,model in enumerate(models):
    print(f"new model {i}")
    for thr in [0.1,0.2,0.01,0.05]:
        try_out_models_for_col_thresh(X_train_wo_grid, X_test_wo_grid,
                                    y_train_c, y_test_c,model,
                                    col='pot_affected_perc',thresh=thr)
#  these truly all did really poor (sad)
# to best one: lgb wiht 0.01 threshold
# mean CV score: 0.009138533564087026 with thresh: 0.01
# R²: 0.0530  RMSE: 6.5564

In [ ]:
for thr in [0.1,0.2,0.01,0.05]:
    try_out_models_for_col_thresh(X_train_wo_grid, X_test_wo_grid,
                                y_train_c, y_test_c,RandomForestRegressor(random_state=42,n_jobs=-1),
                                col='pot_affected_perc',thresh=thr)

# mean CV score: -0.13167066953625714 with thresh: 0.1
# R²: -0.0167  RMSE: 5.9527
# mean CV score: -0.2688381974810109 with thresh: 0.2
# R²: 0.0419  RMSE: 5.4197
# mean CV score: -0.040010856776140004 with thresh: 0.01
# R²: 0.0527  RMSE: 6.5573
# mean CV score: -0.08629717701819159 with thresh: 0.05
# R²: 0.0073  RMSE: 6.4420

In [ ]:
for thr in [0.1,0.2,0.3,0.4]:
        try_out_models_for_col_thresh(X_train_wo_grid, X_test_wo_grid,
                                    y_train_c, y_test_c,RandomForestRegressor(random_state=42,n_jobs=-1),
                                    col='perc_available_upscaled_new',thresh=thr)
        
# mean CV score: -0.0954766225092442 with thresh: 0.1
# R²: 0.0673  RMSE: 6.1329
# mean CV score: -0.08148346285382739 with thresh: 0.2
# R²: -0.0519  RMSE: 5.3271
# mean CV score: -2.202591635135985 with thresh: 0.3
# R²: -9.7862  RMSE: 0.2364
# mean CV score: -21512.35410287419 with thresh: 0.4
# R²: -5.6285  RMSE: 0.0002

In [ ]:
models = [ExtraTreesRegressor(random_state=42,n_jobs=-1),lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0),
          HistGradientBoostingRegressor(random_state=42,verbose=0),BaggingRegressor(random_state=42, n_jobs=-1, verbose=0),
          GradientBoostingRegressor(random_state=42,verbose=0),xgb.XGBRegressor(random_state=42)]
for i,model in enumerate(models):
    print(f"new model {i}")
    for thr in [0.1,0.2,0.3,0.4]:
        try_out_models_for_col_thresh(X_train_wo_grid, X_test_wo_grid,
                                    y_train_c, y_test_c,model,
                                    col='perc_available_upscaled_new',thresh=thr)

#  these truly all did really poor (sad)
# mean CV score: -0.0954766225092442 with thresh: 0.1
# R²: 0.0673  RMSE: 6.1329
# mean CV score: -0.08148346285382739 with thresh: 0.2
# R²: -0.0519  RMSE: 5.3271
# mean CV score: -2.202591635135985 with thresh: 0.3
# R²: -9.7862  RMSE: 0.2364
# mean CV score: -21512.35410287419 with thresh: 0.4
# R²: -5.6285  RMSE: 0.0002

None of these really preformed well, so what we want to do now is to try to add some weighting to see if this helps at all:

In [ ]:
# this should give a higher weight to high-availability cells rather than dropping low ones
# as an alternagtive to what was done before
sample_weights = np.where(
    X_train_wo_grid.loc[X_train_c_simp_selec_wo.index, 'perc_available_upscaled_new'] >= 0.05,
    2.0, 0.5
)
print_consistent_results(best_lgbm_wo_grid, X_train_c_simp_selec_wo, y_train_c,
                         X_test_c_simp_select_wo, y_test_c, cv=5,sample_weight=sample_weights)

#base
# Cross-validation R²:   0.1109 ± 0.0412
# Cross-validation RMSE: 4.6701 ± 0.4136
# Test Set R²:           0.1089
# Test Set RMSE:         4.9296

# 0.05, 3.0, 1.0
# Cross-validation R²:   0.1118 ± 0.0419
# Cross-validation RMSE: 4.6655 ± 0.3907
# Test Set R²:           0.1123
# Test Set RMSE:         4.9202

# 0.1, 3.0, 1.0 - not good
# 0.1, 2.0, 1.0 - not as good
# 0.2, 2.0, 1.0 - not as good
# 0.1, 1.0, 0.5 - not as good
# 0.05, 2.0, 0.5 - not as good

# 0.2,1.0, 0.5
# Cross-validation R²:   0.1122 ± 0.0386
# Cross-validation RMSE: 4.6657 ± 0.3984
# Test Set R²:           0.1048
# Test Set RMSE:         4.9409

# 0.05, 1.0, 0.5
# Cross-validation R²:   0.1124 ± 0.0466
# Cross-validation RMSE: 4.6639 ± 0.3969
# Test Set R²:           0.1107
# Test Set RMSE:         4.9246

# 0.05, 2.0, 1.0
# Cross-validation R²:   0.1120 ± 0.0470
# Cross-validation RMSE: 4.6647 ± 0.3963
# Test Set R²:           0.1180
# Test Set RMSE:         4.9042


In [ ]:
# this should give a higher weight to high-availability cells rather than dropping low ones
# as an alternagtive to what was done before
sample_weights = np.where(X_train_wo_grid['perc_available_upscaled_new'] >= 0.05, 5.0, 1.0)

# HistGradientBoostingRegressor
# LGBMRegressor
best_hgb_wo_grid.fit(X_train_wo_grid[cols_wo_grid_hgb], y_train_c, sample_weight=sample_weights)
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid[cols_wo_grid_hgb], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5,sample_weight=sample_weights)

# 0.15,2.0,1.0
# Cross-validation R²:   0.1176 ± 0.0234
# Cross-validation RMSE: 4.6510 ± 0.3751
# Test Set R²:           0.0876
# Test Set RMSE:         4.9880

# 0.05,3.0,1.0
# Cross-validation R²:   0.1084 ± 0.0349
# Cross-validation RMSE: 4.6750 ± 0.3884
# Test Set R²:           0.0896
# Test Set RMSE:         4.9827

# 0.1,3.0,1.0
# Cross-validation R²:   0.1072 ± 0.0219
# Cross-validation RMSE: 4.6782 ± 0.3733
# Test Set R²:           0.0940
# Test Set RMSE:         4.9704

# 0.2,3.0,1.0
# Cross-validation R²:   0.1123 ± 0.0226
# Cross-validation RMSE: 4.6650 ± 0.3740``
# Test Set R²:           0.0916
# Test Set RMSE:         4.9770

We're seeing marginal gains with the weighting here, so just trying to scalar here (even though it didn't make improvements in the gridded)

In [ ]:
scaler = StandardScaler()
pipeline_wo_1 = Pipeline([('s', scaler), ('m', best_hgb_wo_grid)])

print_consistent_results(pipeline_wo_1, X_train_wo_grid[cols_wo_grid_hgb], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# Cross-validation R²:   0.1184 ± 0.0259
# Cross-validation RMSE: 4.6483 ± 0.3693
# Test Set R²:           0.0951
# Test Set RMSE:         4.9675

In [ ]:
binary_cols = [c for c in cols_wo_grid_hgb if X_train_wo_grid[c].nunique() == 2]
continuous_cols = [c for c in cols_wo_grid_hgb if c not in binary_cols]

preprocessor = ColumnTransformer([
    ('scale', PowerTransformer(), continuous_cols),
    ('passthrough', 'passthrough', binary_cols)
])

pipeline_wo_2 = Pipeline([('pre', preprocessor), ('m', best_hgb_wo_grid)])

print_consistent_results(pipeline_wo_2, X_train_wo_grid[cols_wo_grid_hgb], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# Cross-validation R²:   0.1177 ± 0.0262
# Cross-validation RMSE: 4.6501 ± 0.3711
# Test Set R²:           0.0951
# Test Set RMSE:         4.9675

In [ ]:
scaler = MinMaxScaler()
pipeline_wo_2 = Pipeline([('s',scaler),('m',best_hgb_wo_grid)])

print_consistent_results(pipeline_wo_2, X_train_wo_grid[cols_wo_grid_hgb], y_train_c, 
                         X_test_wo_grid[cols_wo_grid_hgb], y_test_c, cv=5)

# Cross-validation R²:   0.1177 ± 0.0262
# Cross-validation RMSE: 4.6501 ± 0.3711
# Test Set R²:           0.0951
# Test Set RMSE:         4.9675

In [ ]:
import scipy.stats as stats

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# original
axes[0].hist(y_train_c, bins=50)
axes[0].set_title("Original")

# log1p
axes[1].hist(np.log1p(y_train_c), bins=50)
axes[1].set_title("log1p")

# sqrt
axes[2].hist(np.sqrt(y_train_c), bins=50)
axes[2].set_title("sqrt")

plt.tight_layout()
plt.savefig("target_distributions.png")

# also check skewness
print(f"Original skew:  {stats.skew(y_train_c):.3f}")
print(f"log1p skew:     {stats.skew(np.log1p(y_train_c)):.3f}")
print(f"sqrt skew:      {stats.skew(np.sqrt(y_train_c)):.3f}")

Original skew:  11.093

log1p skew:     6.309

sqrt skew:      7.399

![distribution from above](../figures/y_distribution_base_for_notebook.png)

going to add in the logistic regression results data that were done in the logistic regression notebook, and see if this helps the results:

In [ ]:
#confusion matrix cmap
same_scale_color_map = mcolors.LinearSegmentedColormap.from_list(
    "my_cmap",
    ['#D6F1FF', '#33BBFF', '#0088ce', '#005F8F', '#001B29']
)

# #we need to make our target varaible from a % impact into a binary variable
y_train_new = (y_train_c > 0).astype(int)
y_test_new = (y_test_c > 0).astype(int)

In [ ]:
logistic_regression_model = make_pipeline(StandardScaler(), 
                LogisticRegression(max_iter=1000,class_weight="balanced",C=1, l1_ratio=0, solver="liblinear"))

logistic_regression_model.fit(X_train_wo_grid, y_train_new)

probabilities = logistic_regression_model.predict_proba(X_test_wo_grid)[:, 1]  # probability of being impacted
# predictions_updated = (probabilities >= 0.669109).astype(int)
preds = logistic_regression_model.predict(X_test_wo_grid)
preds_2 = (probabilities >= 0.65).astype(int)

results = pd.DataFrame({
    "impacted":    preds,
    "probability": probabilities.round(3),
    "actual": y_test_new.values
})
results["impacted"] = results["impacted"].map({1: "Impacted", 0: "Not Impacted"})

# "stratified" respects class imbalance, better than "most_frequent" for imbalanced data
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train_wo_grid, y_train_new)

dummy_preds = dummy.predict(X_test_wo_grid)
dummy_probs = dummy.predict_proba(X_test_wo_grid)[:, 1]

Use the following to get the threshold for our new assignments:

In [ ]:
results.groupby("actual")["probability"].median()

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.65, 0.7]

print(f"{'Threshold':<12} {'Precision':<11} {'Recall':<10} {'F1':<9} {'AUC':<10}")
print("-" * 52)
for t in thresholds:
    preds_t = (probabilities >= t).astype(int)
    print(f"{t:<12} "
          f"{precision_score(y_test_new, preds_t):<12.3f}"
          f"{recall_score(y_test_new, preds_t):<10.3f}"
          f"{f1_score(y_test_new, preds_t):<10.3f}"
          f"{roc_auc_score(y_test_new, preds_t):<10.3f}")
    

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

cm_dummy = confusion_matrix(y_test_new, dummy_preds)
ConfusionMatrixDisplay(confusion_matrix=cm_dummy, display_labels=["Not Impacted", "Impacted"]).plot(
    cmap=same_scale_color_map, ax=axes[0]
)
axes[0].set_title("Dummy Classifier — Confusion Matrix")

cm = confusion_matrix(y_test_new, preds)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Impacted", "Impacted"]).plot(
    cmap=same_scale_color_map, ax=axes[1]
)
axes[1].set_title("Logistic Regression — Confusion Matrix")

cm = confusion_matrix(y_test_new, preds_2)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Impacted", "Impacted"]).plot(
    cmap=same_scale_color_map, ax=axes[2]
)
axes[2].set_title("Logistic Regression (New Threshold) — Confusion Matrix")

fig.suptitle(f'Confusion Matrices Comparing Dummy to Logistic Regression Results', fontsize=18, fontweight='bold',y=1.01)
plt.tight_layout()
plt.show()

![distribution from above](../figures/confusion_matrices_finalll.png)

In [ ]:
print(f"{'':<13}{'Dummy':<11} {'Base':<8} {'New Threshold':<10}")
print("-" * 47)
print(f"{f"F1 Score":<12} "
          f"{f1_score(y_test_new, dummy_preds):<12.3f}"
          f"{f1_score(y_test_new, preds):<10.3f}"
          f"{f1_score(y_test_new, preds_2):<10.3f}")
print(f"{f"AUC Score":<12} "
          f"{roc_auc_score(y_test_new, dummy_preds):<12.3f}"
          f"{roc_auc_score(y_test_new, preds):<10.3f}"
          f"{roc_auc_score(y_test_new, preds_2):<10.3f}")

#              Dummy       Base     New Threshold
# -----------------------------------------------
# F1 Score     0.127       0.382     0.410     
# AUC Score    0.507       0.716     0.690  

In [ ]:
auc = roc_auc_score(y_test_new, preds_2)
fpr, tpr, _ = roc_curve(y_test_new, probabilities)

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}", color='#7FB646')
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

![distribution from above](../figures/roc_curve_final_preds.png)

Add the predictions to the data to filter and fix the next model:

In [ ]:
# # get the probability predicted
prob_train = logistic_regression_model.predict_proba(X_train_wo_grid)[:, 1]
prob_test = logistic_regression_model.predict_proba(X_test_wo_grid)[:, 1]

#assign those to a column to filter our data on
logit_train = (prob_train >= 0.65).astype(int)
logit_test = (prob_test >= 0.65).astype(int)

# make a mask out of the above
nonzero_mask_train = logit_train == 1
nonzero_mask_test  = logit_test  == 1

X_train_li = X_train_wo_grid[nonzero_mask_train]
y_train_li = y_train_c[nonzero_mask_train]
X_test_li = X_test_wo_grid[nonzero_mask_test]
y_test_li = y_test_c[nonzero_mask_test]

Now, lets filter our model data and fit a model:

In [ ]:
best_hgb_wo_grid.fit(X_train_li[cols_wo_grid_hgb], y_train_li)
print_consistent_results(best_hgb_wo_grid, X_train_li[cols_wo_grid_hgb], y_train_li, 
                         X_test_li[cols_wo_grid_hgb], y_test_li, cv=5)

# Cross-validation R²:   0.0705 ± 0.0532
# Cross-validation RMSE: 9.1309 ± 0.8891
# Test Set R²:           0.0557
# Test Set RMSE:         9.5750

The results weren't looking good here, trying to run some baselines to compare how our other original baselines compare and if it's best to try a new model:

In [ ]:
# trying three random other models to see how they compare

# lightgbm
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
print_consistent_results(rf_b_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
print_consistent_results(ex_b_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
print_consistent_results(lgb_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
print_consistent_results(hgbr_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
print_consistent_results(br_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for GradientBoostingRegressor:")
gbr_model = GradientBoostingRegressor(random_state=42,verbose=0)
print_consistent_results(gbr_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
print_consistent_results(xgb_b_model, X_train_li, y_train_li, X_test_li, y_test_li, cv=5)

This was run for a threshold of: 0.5, 0.60, 0.65. None of the results preformed as well as our original baselines. And seemed too far off to be improved with the other work we have done.

Since, that didn't make this any better, we're going to just add in the prediction as part of the data without filtering anything out:

In [ ]:
# # get the probability predicted - from before:
# prob_train = logistic_regression_model.predict_proba(X_train_wo_grid)[:, 1]
# prob_test = logistic_regression_model.predict_proba(X_test_wo_grid)[:, 1]

# make copy of dataframs of predictors
X_train_wo_grid_w_pred = X_train_wo_grid.copy()
X_test_wo_grid_w_pred = X_test_wo_grid.copy()

# add a column to dataframes
X_train_wo_grid_w_pred['logit_prob'] = prob_train
X_test_wo_grid_w_pred['legit_prob'] = prob_test

In [ ]:
X_train_wo_grid.drop(columns=['log_pred_impacted'],inplace=True)
X_test_wo_grid.drop(columns=['log_pred_impacted'],inplace=True)
# log_pred_impacted

Going to do some testing in concession with all the predictions data:

In [ ]:
mean_columns = [col for col in X_train_wo_grid.columns if 'mean' in col]
median_columns = [col for col in X_train_wo_grid.columns if 'median' in col]

X_train_wo_grid_medians = X_train_wo_grid_w_pred.drop(columns=mean_columns)
X_test_wo_grid_medians = X_test_wo_grid_w_pred.drop(columns=mean_columns)

X_train_wo_grid_means = X_train_wo_grid_w_pred.drop(columns=median_columns)
X_test_wo_grid_means = X_test_wo_grid_w_pred.drop(columns=median_columns)


Ok, now try the baseline on all of these combinations (3 combos):

In [ ]:
best_lgbm_tester(best_lgbm_wo_grid,X_train_wo_grid_medians,y_train_c,X_test_wo_grid_medians,y_test_c,thresh=0.9)

# Cross-validation R²:   0.0889 ± 0.0244
# Cross-validation RMSE: 4.7256 ± 0.3744
# Test Set R²:           0.0833
# Test Set RMSE:         4.9997
# Cross-validation R²:   0.0947 ± 0.0353
# Cross-validation RMSE: 4.7088 ± 0.3673
# Test Set R²:           0.0888
# Test Set RMSE:         4.9847

In [ ]:
best_lgbm_tester(best_lgbm_wo_grid,X_train_wo_grid_means,y_train_c,X_test_wo_grid_means,y_test_c,thresh=0.9)

# Cross-validation R²:   0.1044 ± 0.0375
# Cross-validation RMSE: 4.6861 ± 0.3995
# Test Set R²:           0.0959
# Test Set RMSE:         4.9653
# Cross-validation R²:   0.0947 ± 0.0389
# Cross-validation RMSE: 4.7076 ± 0.3563
# Test Set R²:           0.1057
# Test Set RMSE:         4.9384

In [ ]:
print_consistent_results(lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0), X_train_wo_grid_medians,y_train_c,
                         X_test_wo_grid_medians,y_test_c, cv=5)

# Cross-validation R²:   0.0595 ± 0.0402
# Cross-validation RMSE: 4.7971 ± 0.3469
# Test Set R²:           0.0963
# Test Set RMSE:         4.9644

In [ ]:
print_consistent_results(lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0), X_train_wo_grid_means,y_train_c,
                         X_test_wo_grid_means,y_test_c, cv=5)

# Cross-validation R²:   0.0724 ± 0.0435
# Cross-validation RMSE: 4.7646 ± 0.3581
# Test Set R²:           0.0852
# Test Set RMSE:         4.9946

quick basic test with values:

In [ ]:
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid_medians,y_train_c,
                         X_test_wo_grid_medians,y_test_c, cv=5)

# Cross-validation R²:   0.0920 ± 0.0333
# Cross-validation RMSE: 4.7155 ± 0.3588
# Test Set R²:           0.1055
# Test Set RMSE:         4.9390

In [ ]:
print_consistent_results(best_hgb_wo_grid, X_train_wo_grid_means, y_train_c,
                         X_test_wo_grid_means,y_test_c, cv=5)

# Cross-validation R²:   0.0999 ± 0.0272
# Cross-validation RMSE: 4.6975 ± 0.3819
# Test Set R²:           0.0882
# Test Set RMSE:         4.9863

In [ ]:
print_consistent_results(HistGradientBoostingRegressor(random_state=42), X_train_wo_grid_medians,y_train_c,
                         X_test_wo_grid_medians,y_test_c, cv=5)

# Cross-validation R²:   0.0711 ± 0.0385
# Cross-validation RMSE: 4.7677 ± 0.3472
# Test Set R²:           0.1036
# Test Set RMSE:         4.9442

In [ ]:
print_consistent_results(HistGradientBoostingRegressor(random_state=42), X_train_wo_grid_means,y_train_c,
                         X_test_wo_grid_means,y_test_c, cv=5)

# Cross-validation R²:   0.0800 ± 0.0274
# Cross-validation RMSE: 4.7482 ± 0.3744
# Test Set R²:           0.0771
# Test Set RMSE:         5.0167

In [118]:
q25 = y_train_c.quantile(0.25)
q75 = y_train_c.quantile(0.75)

tail_mask = (y_train_c < q25) | (y_train_c > q75)
X_tail = X_train_c[tail_mask]
y_tail = y_train_c[tail_mask]

n_repeats = 5
noise_X = 0.01
noise_y = 0.01 * y_train_c.std()

X_augmented = pd.concat([X_tail + np.random.normal(0, noise_X, X_tail.shape) for _ in range(n_repeats)])
y_augmented = pd.concat([y_tail + np.random.normal(0, noise_y, y_tail.shape) for _ in range(n_repeats)])

X_train_over = pd.concat([X_train_c, X_augmented]).reset_index(drop=True)
y_train_over = pd.concat([y_train_c, y_augmented]).reset_index(drop=True)

print(f"Original size: {len(X_train_c)}, Oversampled size: {len(X_train_over)}")

# X_test_c, y_test_c

Original size: 24448, Oversampled size: 38168


In [ ]:
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_over, y_train_over)

scores = cross_val_score(rf_b_model, X_train_over, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_rfb = rf_b_model.predict(X_test_c)
test_r2_rfb = r2_score(y_test_c, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_over, y_train_over)

scores = cross_val_score(ex_b_model, X_train_over, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_exb = ex_b_model.predict(X_test_c)
test_r2_exb = r2_score(y_test_c, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")

# print("for LGBM:")
# lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
# lgb_model.fit(X_train_over, y_train_over)

# scores = cross_val_score(lgb_model, X_train_over, y_train_over, cv=5, scoring='r2')
# print(f"mean CV score: {scores.mean():.4f}")

# y_pred_lgb = lgb_model.predict(X_test_c)
# test_r2_lgb = r2_score(y_test_c, y_pred_lgb)
# print(f"Test Set R²: {test_r2_lgb:.4f}")

# print("for HistGradientBoostingRegressor:")
# hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
# hgbr_model.fit(X_train_over, y_train_over)

# scores = cross_val_score(hgbr_model, X_train_over, y_train_over, cv=5, scoring='r2',verbose=0)
# print(f"mean CV score: {scores.mean():.4f}")

# y_pred_hgbr = hgbr_model.predict(X_test_c)
# test_r2_hgbr = r2_score(y_test_c, y_pred_hgbr)
# print(f"Test Set R²: {test_r2_hgbr:.4f}")

# print("for BaggingRegressor:")
# br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
# br_model.fit(X_train_over, y_train_over)

# scores = cross_val_score(br_model, X_train_over, y_train_over, cv=5, scoring='r2')
# print(f"mean CV score: {scores.mean():.4f}")

# y_pred_br = br_model.predict(X_test_c)
# test_r2_br = r2_score(y_test_c, y_pred_br)
# print(f"Test Set R²: {test_r2_br:.4f}")

# print("for GradientBoostingRegressor:")
# gbr_model = GradientBoostingRegressor(random_state=42,verbose=0)
# gbr_model.fit(X_train_over, y_train_over)

# scores = cross_val_score(gbr_model, X_train_over, y_train_over, cv=5, scoring='r2')
# print(f"mean CV score: {scores.mean():.4f}")

# y_pred_gbr = gbr_model.predict(X_test_c)
# test_r2_gbr = r2_score(y_test_c, y_pred_gbr)
# print(f"Test Set R²: {test_r2_gbr:.4f}")

# print("for XGBRegressor:")
# xgb_b_model = xgb.XGBRegressor(random_state=42)
# xgb_b_model.fit(X_train_over, y_train_over)

# scores = cross_val_score(xgb_b_model, X_train_over, y_train_over, cv=5, scoring='r2')
# print(f"mean CV score: {scores.mean():.4f}")

# y_pred_xgbb = xgb_b_model.predict(X_test_c)
# test_r2_xgbb = r2_score(y_test_c, y_pred_xgbb)
# print(f"Test Set R²: {test_r2_xgbb:.4f}")

#results for repeats 2
# for RandomForest:
# mean CV score: 0.3274
# Test Set R²: 0.3648
# for ExtraTrees:
# mean CV score: 0.3265
# Test Set R²: 0.3095
# for LGBM:
# mean CV score: 0.3108
# Test Set R²: 0.2786
# for HistGradientBoostingRegressor:
# mean CV score: 0.2797
# Test Set R²: 0.3013
# for BaggingRegressor:
# mean CV score: 0.2854
# Test Set R²: 0.2794
# for GradientBoostingRegressor:
# mean CV score: 0.1395
# Test Set R²: 0.1313
# for XGBRegressor:
# mean CV score: 0.2930
# Test Set R²: 0.2961
# for SVR:
# mean CV score: -0.0296
# Test R²: -0.0065

#results for repeats: 4
# for RandomForest:
# mean CV score: 0.3742
# Test Set R²: 0.3613
# for ExtraTrees:
# mean CV score: 0.3189
# Test Set R²: 0.3117
# for LGBM:
# mean CV score: 0.3953
# Test Set R²: 0.2678
# for HistGradientBoostingRegressor:
# mean CV score: 0.3729
# Test Set R²: 0.2490
# for BaggingRegressor:
# mean CV score: 0.3108
# Test Set R²: 0.2747
# for GradientBoostingRegressor:
# mean CV score: 0.2023
# Test Set R²: 0.1238
# for XGBRegressor:
# mean CV score: 0.3776
# Test Set R²: 0.2390

#results for repeats: 5
# for RandomForest:
# mean CV score: 0.4389
# Test Set R²: 0.4159
# for ExtraTrees:
# mean CV score: 0.3656
# Test Set R²: 0.3217
# for LGBM:
# mean CV score: 0.4174
# Test Set R²: 0.2916

#results for repeats: 6
# for RandomForest:
# mean CV score: 0.4919
# Test Set R²: 0.4055
# for ExtraTrees:
# mean CV score: 0.4006
# Test Set R²: 0.3118


for RandomForest:
mean CV score: 0.4919
Test Set R²: 0.4055
for ExtraTrees:
mean CV score: 0.4006
Test Set R²: 0.3118


without grid:

In [ ]:
tail_mask = (y_train_c < q25) | (y_train_c > q75)
X_tail_wo_grid = X_train_wo_grid[tail_mask]
y_tail = y_train_c[tail_mask]

n_repeats = 5
noise_X = 0.01
noise_y = 0.01 * y_train_c.std()

X_augmented_wo_grid = pd.concat([ss + np.random.normal(0, noise_X, X_tail_wo_grid.shape) for _ in range(n_repeats)])
y_augmented = pd.concat([y_tail + np.random.normal(0, noise_y, y_tail.shape) for _ in range(n_repeats)])

X_train_over_wo_grid = pd.concat([X_train_wo_grid, X_augmented_wo_grid]).reset_index(drop=True)
y_train_over = pd.concat([y_train_c, y_augmented]).reset_index(drop=True)

print(f"Original size: {len(X_train_wo_grid)}, Oversampled size: {len(X_train_over_wo_grid)}")

Original size: 24448, Oversampled size: 38168


In [ ]:
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_over_wo_grid, y_train_over)
scores = cross_val_score(rf_b_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_rfb = rf_b_model.predict(X_test_wo_grid)
test_r2_rfb = r2_score(y_test_c, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")
print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_over_wo_grid, y_train_over)
scores = cross_val_score(ex_b_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_exb = ex_b_model.predict(X_test_wo_grid)
test_r2_exb = r2_score(y_test_c, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")
print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train_over_wo_grid, y_train_over)
scores = cross_val_score(lgb_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_lgb = lgb_model.predict(X_test_wo_grid)
test_r2_lgb = r2_score(y_test_c, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")
print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train_over_wo_grid, y_train_over)
scores = cross_val_score(hgbr_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_hgbr = hgbr_model.predict(X_test_wo_grid)
test_r2_hgbr = r2_score(y_test_c, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")

# for n = 5
# for RandomForest:
# mean CV score: 0.1345
# Test Set R²: -0.0633
# for ExtraTrees:
# mean CV score: 0.0973
# Test Set R²: -0.0909
# for LGBM:
# mean CV score: 0.2045
# Test Set R²: 0.0673
# for HistGradientBoostingRegressor:
# mean CV score: 0.1929
# Test Set R²: 0.0714

for RandomForest:
mean CV score: 0.1345
Test Set R²: -0.0633
for ExtraTrees:
mean CV score: 0.0973
Test Set R²: -0.0909
for LGBM:
mean CV score: 0.2045
Test Set R²: 0.0673
for HistGradientBoostingRegressor:
mean CV score: 0.1929
Test Set R²: 0.0714


In [110]:
for n in [2,4]: 
    tail_mask = (y_train_c < q25) | (y_train_c > q75)
    X_tail_wo_grid = X_train_wo_grid[tail_mask]
    y_tail = y_train_c[tail_mask]

    n_repeats = n
    noise_X = 0.01
    noise_y = 0.01 * y_train_c.std()

    X_augmented_wo_grid = pd.concat([X_tail_wo_grid + np.random.normal(0, noise_X, X_tail_wo_grid.shape) for _ in range(n_repeats)])
    y_augmented = pd.concat([y_tail + np.random.normal(0, noise_y, y_tail.shape) for _ in range(n_repeats)])

    X_train_over_wo_grid = pd.concat([X_train_wo_grid, X_augmented_wo_grid]).reset_index(drop=True)
    y_train_over = pd.concat([y_train_c, y_augmented]).reset_index(drop=True)

    print(f"Original size: {len(X_train_wo_grid)}, Oversampled size: {len(X_train_over_wo_grid)}")

    print(f"\nresults for {n}")

    print("for LGBM:")
    lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
    lgb_model.fit(X_train_over_wo_grid, y_train_over)
    scores = cross_val_score(lgb_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
    print(f"mean CV score: {scores.mean():.4f}")
    y_pred_lgb = lgb_model.predict(X_test_wo_grid)
    test_r2_lgb = r2_score(y_test_c, y_pred_lgb)
    print(f"Test Set R²: {test_r2_lgb:.4f}")
    print("for HistGradientBoostingRegressor:")
    hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
    hgbr_model.fit(X_train_over_wo_grid, y_train_over)
    scores = cross_val_score(hgbr_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
    print(f"mean CV score: {scores.mean():.4f}")
    y_pred_hgbr = hgbr_model.predict(X_test_wo_grid)
    test_r2_hgbr = r2_score(y_test_c, y_pred_hgbr)
    print(f"Test Set R²: {test_r2_hgbr:.4f}")
    print("for BaggingRegressor:")
    br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
    br_model.fit(X_train_over_wo_grid, y_train_over)
    scores = cross_val_score(br_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
    print(f"mean CV score: {scores.mean():.4f}")
    y_pred_br = br_model.predict(X_test_wo_grid)
    test_r2_br = r2_score(y_test_c, y_pred_br)
    print(f"Test Set R²: {test_r2_br:.4f}")
    print("for XGBRegressor:")
    xgb_b_model = xgb.XGBRegressor(random_state=42)
    xgb_b_model.fit(X_train_over_wo_grid, y_train_over)
    scores = cross_val_score(xgb_b_model, X_train_over_wo_grid, y_train_over, cv=5, scoring='r2')
    print(f"mean CV score: {scores.mean():.4f}")
    y_pred_xgbb = xgb_b_model.predict(X_test_wo_grid)
    test_r2_xgbb = r2_score(y_test_c, y_pred_xgbb)
    print(f"Test Set R²: {test_r2_xgbb:.4f}")


Original size: 24448, Oversampled size: 29936

results for 2
for LGBM:


mean CV score: 0.1481
Test Set R²: 0.1071
for HistGradientBoostingRegressor:
mean CV score: 0.1187
Test Set R²: 0.0951
for BaggingRegressor:
mean CV score: 0.0315
Test Set R²: -0.1139
for XGBRegressor:
mean CV score: 0.1124
Test Set R²: 0.0939
Original size: 24448, Oversampled size: 35424

results for 4
for LGBM:
mean CV score: 0.1866
Test Set R²: 0.0895
for HistGradientBoostingRegressor:
mean CV score: 0.1626
Test Set R²: 0.0896
for BaggingRegressor:
mean CV score: 0.0367
Test Set R²: -0.1154
for XGBRegressor:
mean CV score: 0.1684
Test Set R²: 0.0567


Final set is one with the more complicated filling method, which I may just drop, didn't improve results.

In [ ]:
# read in complex filled data
df = pd.read_csv('../data/target/model_training_data_final.csv')
X_train_mc, X_test_mc, y_train_mc, y_test_mc = clean_and_split_data(df,['percent_houses_damaged_5years','percent_houses_damaged'],'percent_houses_damaged',filling_method='more_complex')

In [116]:
q25 = y_train_mc.quantile(0.25)
q75 = y_train_mc.quantile(0.75)

tail_mask = (y_train_mc < q25) | (y_train_mc > q75)
X_tail = X_train_mc[tail_mask]
y_tail = y_train_mc[tail_mask]

n_repeats = 5
noise_X = 0.01
noise_y = 0.01 * y_train_mc.std()

X_augmented = pd.concat([X_tail + np.random.normal(0, noise_X, X_tail.shape) for _ in range(n_repeats)])
y_augmented = pd.concat([y_tail + np.random.normal(0, noise_y, y_tail.shape) for _ in range(n_repeats)])

X_train_over = pd.concat([X_train_mc, X_augmented]).reset_index(drop=True)
y_train_over = pd.concat([y_train_mc, y_augmented]).reset_index(drop=True)

print(f"Original size: {len(X_train_mc)}, Oversampled size: {len(X_train_over)}")

# X_test_c, y_test_c

Original size: 24448, Oversampled size: 38168


In [117]:
print("for RandomForest:")
rf_b_model = RandomForestRegressor(random_state=42, n_jobs=-1,verbose=0);
rf_b_model.fit(X_train_over, y_train_over)

scores = cross_val_score(rf_b_model, X_train_over, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_rfb = rf_b_model.predict(X_test_mc)
test_r2_rfb = r2_score(y_test_mc, y_pred_rfb)
print(f"Test Set R²: {test_r2_rfb:.4f}")

print("for ExtraTrees:")
ex_b_model = ExtraTreesRegressor(random_state=42, n_jobs=-1,verbose=0);
ex_b_model.fit(X_train_over, y_train_over)

scores = cross_val_score(ex_b_model, X_train_over, y_train_over, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")

y_pred_exb = ex_b_model.predict(X_test_mc)
test_r2_exb = r2_score(y_test_mc, y_pred_exb)
print(f"Test Set R²: {test_r2_exb:.4f}")

for RandomForest:
mean CV score: 0.4349
Test Set R²: 0.4100
for ExtraTrees:
mean CV score: 0.3617
Test Set R²: 0.3098


Maybe will try with that a bit more later, but the final version of this we will add in land cover data. One method will include using a perfiltered dataset that only had suburban and urban cells that have been added up, and the other one will filter up and add the landcover type as another few columns.

As well as fine tuning the best prefomring models here, based on previous work.

This will be added to a new notebook.